# Phase 2 — Predictive ML Models
Train XGBoost regression models to forecast, from `data/processed/disruptions_clean.parquet`:
- `production_impact_pct`
- `revenue_loss_usd`
- `full_recovery_days`

Feature sets respect causal/temporal order so later-stage fields don't leak into earlier-stage predictions. Categorical columns use pandas `category` dtype with XGBoost's native categorical support (no one-hot blowup). SHAP/XAI is deferred to a later pass.

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

DATA_PATH = Path("data/processed/disruptions_clean.parquet")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## Load cleaned data

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

CATEGORICAL_COLUMNS = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "disruption_severity",
    "response_type",
]
for column in CATEGORICAL_COLUMNS:
    df[column] = df[column].astype("category")

# XGBoost's categorical path rejects boolean-typed categories, so keep
# these two binary flags as plain 0/1 ints instead of pandas categories.
BOOLEAN_COLUMNS = ["has_backup_supplier", "permanent_supplier_change"]
for column in BOOLEAN_COLUMNS:
    df[column] = df[column].astype("int64")

df.dtypes

Loaded 100,000 rows, 15 columns


disruption_id                  string
disruption_type              category
industry                     category
supplier_tier                category
supplier_region              category
supplier_size                category
has_backup_supplier             int64
disruption_severity          category
production_impact_pct         float64
revenue_loss_usd              float64
response_type                category
response_time_days              int64
partial_recovery_days           int64
full_recovery_days              int64
permanent_supplier_change       int64
dtype: object

## Feature sets
- **production_impact_pct** ← pre-disruption supplier/severity attributes only
- **revenue_loss_usd** ← same base features + `production_impact_pct` (known at analysis time)
- **full_recovery_days** ← base features + response/recovery-stage fields

In [3]:
BASE_FEATURES = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "disruption_severity",
]

TARGET_FEATURES = {
    "production_impact_pct": BASE_FEATURES,
    "revenue_loss_usd": BASE_FEATURES + ["production_impact_pct"],
    "full_recovery_days": BASE_FEATURES + [
        "response_type",
        "response_time_days",
        "partial_recovery_days",
        "permanent_supplier_change",
    ],
}

## Train, evaluate, and save each model

In [4]:
def train_one(target, features):
    data = df[features + [target]].dropna()
    X, y = data[features], data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    model = XGBRegressor(
        tree_method="hist",
        enable_categorical=True,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    metrics = {
        "rmse": float(np.sqrt(mean_squared_error(y_test, preds))),
        "mae": float(mean_absolute_error(y_test, preds)),
        "r2": float(r2_score(y_test, preds)),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "features": features,
    }

    model_path = MODELS_DIR / f"{target}_model.json"
    model.save_model(model_path)

    return metrics


all_metrics = {}
for target, features in TARGET_FEATURES.items():
    print(f"Training model for: {target}")
    all_metrics[target] = train_one(target, features)
    print(f"  RMSE={all_metrics[target]['rmse']:.3f}  "
          f"MAE={all_metrics[target]['mae']:.3f}  "
          f"R2={all_metrics[target]['r2']:.3f}\n")

Training model for: production_impact_pct


  RMSE=9.730  MAE=7.820  R2=0.697

Training model for: revenue_loss_usd


  RMSE=4390124.628  MAE=1616087.249  R2=0.375

Training model for: full_recovery_days


  RMSE=38.170  MAE=26.270  R2=0.551



## Results summary

In [5]:
with open(MODELS_DIR / "metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

summary = pd.DataFrame(all_metrics).T[["rmse", "mae", "r2", "n_train", "n_test"]]
print(f"Saved metrics to {MODELS_DIR / 'metrics.json'}")
summary

Saved metrics to models\metrics.json


,rmse,mae,r2,n_train,n_test
production_impact_pct,9.729913,7.819925,0.697428,80000,20000
revenue_loss_usd,4390124.628029,1616087.248908,0.375309,80000,20000
full_recovery_days,38.169696,26.270029,0.551005,80000,20000
